![clothing_classification](clothing_classification.png)


Fashion Forward is a new AI-based e-commerce clothing retailer.
They want to use image classification to automatically categorize new product listings, making it easier for customers to find what they're looking for. It will also assist in inventory management by quickly sorting items.

As a data scientist tasked with implementing a garment classifier, your primary objective is to develop a machine learning model capable of accurately categorizing images of clothing items into distinct garment types such as shirts, trousers, shoes, etc.

Project Instructions

Automate product tagging for the e-commerce store using CNNs.

    Once trained (keeping the epochs to 1 or 2 to keep the run time down), store your predictions on the test set in a list named predictions.

    Calculate the accuracy, and per-class precision and recall for your classifier based on the predictions obtained. Store your metrics in variables named accuracy, precision, and recall. Use lists of the appropriate length for the precision and recall.

In [24]:
# Run the cells below first

In [25]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy, Precision, Recall

In [26]:
# Load datasets
from torchvision import datasets
import torchvision.transforms as transforms

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

In [27]:
# Start coding here
# Use as many cells as you need

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ClothingCNN(nn.Module):
    def __init__(self):
        super(ClothingCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32*7*7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [29]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ClothingCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

In [21]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.numpy())


In [22]:
from torchmetrics import Accuracy, Precision, Recall

# Convert to tensors
preds_tensor = torch.tensor(predictions)
true_tensor = torch.tensor(true_labels)

# Overall accuracy
accuracy_metric = Accuracy(task="multiclass", num_classes=10)
accuracy = accuracy_metric(preds_tensor, true_tensor).item()

# Per-class precision and recall
precision_metric = Precision(task="multiclass", average=None, num_classes=10)
recall_metric = Recall(task="multiclass", average=None, num_classes=10)

precision = precision_metric(preds_tensor, true_tensor).tolist()
recall = recall_metric(preds_tensor, true_tensor).tolist()

print("Accuracy:", accuracy)
print("Precision per class:", precision)
print("Recall per class:", recall)


Accuracy: 0.8838000297546387
Precision per class: [0.7637931108474731, 0.9918450713157654, 0.8513215780258179, 0.9060887694358826, 0.7759562730789185, 0.9775739312171936, 0.7129735946655273, 0.9272727370262146, 0.9700897336006165, 0.9695122241973877]
Recall per class: [0.8859999775886536, 0.9729999899864197, 0.7730000019073486, 0.878000020980835, 0.8519999980926514, 0.9589999914169312, 0.6209999918937683, 0.968999981880188, 0.9729999899864197, 0.9539999961853027]
